In [3]:
from datasets import load_dataset

dataset = load_dataset("chengxuphd/liar2")
print(dataset)

C:\Users\ADMIN\anaconda3\envs\variable\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ADMIN\.cache\huggingface\hub\datasets--chengxuphd--liar2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Generating test split: 100%|██████████| 2296/2296 [00:00<00:00, 8339.18 examples/s]


DatasetDict({
    train: Dataset({
        features: ['id', 'label', 'statement', 'date', 'subject', 'speaker', 'speaker_description', 'state_info', 'true_counts', 'mostly_true_counts', 'half_true_counts', 'mostly_false_counts', 'false_counts', 'pants_on_fire_counts', 'context', 'justification'],
        num_rows: 18369
    })
    validation: Dataset({
        features: ['id', 'label', 'statement', 'date', 'subject', 'speaker', 'speaker_description', 'state_info', 'true_counts', 'mostly_true_counts', 'half_true_counts', 'mostly_false_counts', 'false_counts', 'pants_on_fire_counts', 'context', 'justification'],
        num_rows: 2297
    })
    test: Dataset({
        features: ['id', 'label', 'statement', 'date', 'subject', 'speaker', 'speaker_description', 'state_info', 'true_counts', 'mostly_true_counts', 'half_true_counts', 'mostly_false_counts', 'false_counts', 'pants_on_fire_counts', 'context', 'justification'],
        num_rows: 2296
    })
})


In [4]:
import os
print(os.getcwd())

C:\Users\ADMIN\NLP Project 4


In [5]:
print(dataset['train'][0])

{'id': 13847, 'label': 5, 'statement': '90 percent of Americans "support universal background checks" for gun purchases.', 'date': 'October 2, 2017', 'subject': 'government regulation;polls and public opinion;guns', 'speaker': 'chris abele', 'speaker_description': 'Chris Abele is Milwaukee County Executive, a position he won in an April 2011 special election to finish out the final year of the term of Scott Walker, who was elected governor in November 2010. The election was the first attempt at political office for Abele, a Milwaukee philanthropist and business owner.\r\nThe office is nonpartisan, but Abele has indicated he is a Democrat.', 'state_info': 'wisconsin', 'true_counts': 1, 'mostly_true_counts': 4, 'half_true_counts': 5, 'mostly_false_counts': 3, 'false_counts': 5, 'pants_on_fire_counts': 2, 'context': 'a tweet', 'justification': '"Universal" is the term for background checks to be done on every gun sale. We found support for that policy at 94 percent in the latest national 

In [6]:
import pandas as pd

train_df = pd.DataFrame(dataset['train'])
print(train_df['label'].value_counts().sort_index())

print(train_df['statement'].isnull().sum())
print(train_df['label'].isnull().sum())

label
0    2425
1    5284
2    2882
3    2967
4    2743
5    2068
Name: count, dtype: int64
0
0


In [7]:
text_lengths = [len(s.split()) for s in dataset['train']['statement']]
print("Statement length - min:", min(text_lengths), "max:", max(text_lengths), "avg:", sum(text_lengths)/len(text_lengths))

Statement length - min: 2 max: 79 avg: 17.696826174533182


In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

X_train = dataset['train']['statement']
y_train = dataset['train']['label']
X_test = dataset['test']['statement']
y_test = dataset['test']['label']

vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print(X_train_tfidf.shape)
print(X_test_tfidf.shape)

(18369, 5000)
(2296, 5000)


In [9]:
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train_tfidf, y_train)

y_pred = lr_model.predict(X_test_tfidf)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.3201219512195122
              precision    recall  f1-score   support

           0       0.38      0.22      0.28       303
           1       0.38      0.62      0.47       660
           2       0.23      0.17      0.19       360
           3       0.27      0.23      0.25       371
           4       0.23      0.22      0.23       343
           5       0.30      0.15      0.20       259

    accuracy                           0.32      2296
   macro avg       0.30      0.27      0.27      2296
weighted avg       0.31      0.32      0.30      2296



In [10]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np

tokenizer = Tokenizer(num_words=5000, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

max_len = 40
X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding='post', truncating='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len, padding='post', truncating='post')

y_train_arr = np.array(y_train)
y_test_arr = np.array(y_test)

print(X_train_pad.shape)
print(X_test_pad.shape)

(18369, 40)
(2296, 40)


In [11]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, GlobalMaxPooling1D, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

cnn_model = Sequential([
    Embedding(input_dim=5000, output_dim=128),
    Conv1D(128, 5, activation='relu'),
    GlobalMaxPooling1D(),
    Dense(64, activation='relu'),
    Dropout(0.5),
    Dense(6, activation='softmax')
])

cnn_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

early_stop = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)

history = cnn_model.fit(
    X_train_pad, y_train_arr,
    epochs=10,
    batch_size=64,
    validation_split=0.1,
    callbacks=[early_stop]
)

Epoch 1/10
259/259 ━━━━━━━━━━━━━━━━━━━━ 20s 46ms/step - accuracy: 0.2929 - loss: 1.7094 - val_accuracy: 0.3136 - val_loss: 1.6111
Epoch 2/10
259/259 ━━━━━━━━━━━━━━━━━━━━ 10s 39ms/step - accuracy: 0.3420 - loss: 1.5713 - val_accuracy: 0.3261 - val_loss: 1.5745
Epoch 3/10
259/259 ━━━━━━━━━━━━━━━━━━━━ 9s 35ms/step - accuracy: 0.4271 - loss: 1.4009 - val_accuracy: 0.3239 - val_loss: 1.6120
Epoch 4/10
259/259 ━━━━━━━━━━━━━━━━━━━━ 9s 35ms/step - accuracy: 0.5462 - loss: 1.1567 - val_accuracy: 0.3195 - val_loss: 1.7264


In [12]:
test_loss, test_acc = cnn_model.evaluate(X_test_pad, y_test_arr)
print("Test accuracy:", test_acc)

72/72 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.3288 - loss: 1.5755
Test accuracy: 0.328832745552063


In [14]:
cnn_model.save('cnn_model.keras')

import pickle
with open('tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)

In [15]:
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification

bert_tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

def tokenize_function(examples):
    return bert_tokenizer(examples['statement'], padding='max_length', truncation=True, max_length=40)

tokenized_train = dataset['train'].map(tokenize_function, batched=True)
tokenized_test = dataset['test'].map(tokenize_function, batched=True)

bert_model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=6)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 835.22it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [16]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir='./bert_results',
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=bert_model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
)

In [17]:
trainer.train()

C:\Users\ADMIN\anaconda3\envs\variable\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,1.570979,1.532091
2,1.378314,1.540516


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.03it/s]
C:\Users\ADMIN\anaconda3\envs\variable\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.91it/s]


TrainOutput(global_step=2298, training_loss=1.48845188549231, metrics={'train_runtime': 12779.1551, 'train_samples_per_second': 2.875, 'train_steps_per_second': 0.18, 'total_flos': 380229253634880.0, 'train_loss': 1.48845188549231, 'epoch': 2.0})

In [18]:
from sklearn.metrics import accuracy_score, classification_report

predictions = trainer.predict(tokenized_test)
y_pred = predictions.predictions.argmax(axis=1)
y_true = tokenized_test['label']

print("Accuracy:", accuracy_score(y_true, y_pred))
print(classification_report(y_true, y_pred))

C:\Users\ADMIN\anaconda3\envs\variable\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Accuracy: 0.3397212543554007
              precision    recall  f1-score   support

           0       0.54      0.10      0.17       303
           1       0.41      0.69      0.52       660
           2       0.27      0.12      0.17       360
           3       0.28      0.27      0.27       371
           4       0.25      0.43      0.31       343
           5       0.00      0.00      0.00       259

    accuracy                           0.34      2296
   macro avg       0.29      0.27      0.24      2296
weighted avg       0.31      0.34      0.29      2296



C:\Users\ADMIN\anaconda3\envs\variable\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\ADMIN\anaconda3\envs\variable\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\ADMIN\anaconda3\envs\variable\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", res

In [20]:
# Logistic Regression predictions
y_pred_lr = lr_model.predict(X_test_tfidf)

# CNN predictions
y_pred_cnn = cnn_model.predict(X_test_pad).argmax(axis=1)

# DistilBERT predictions (already have these, just renaming for clarity)
y_pred_bert = predictions.predictions.argmax(axis=1)

72/72 ━━━━━━━━━━━━━━━━━━━━ 3s 20ms/step


In [21]:
import os
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, roc_curve, auc
from sklearn.preprocessing import label_binarize

plot_dir = "plots"
os.makedirs(plot_dir, exist_ok=True)

labels = ['Pants-fire', 'False', 'Barely-true', 'Half-true', 'Mostly-true', 'True']
n_classes = 6

#Confusion Matrices
def save_confusion_matrix(y_true, y_pred, title, filename):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(9, 7))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title(title)
    plt.savefig(os.path.join(plot_dir, filename), bbox_inches='tight', dpi=150)
    plt.close()

save_confusion_matrix(y_test, y_pred_lr, 'Logistic Regression Confusion Matrix', 'lr_confusion_matrix.png')
save_confusion_matrix(y_test_arr, y_pred_cnn, 'CNN Confusion Matrix', 'cnn_confusion_matrix.png')
save_confusion_matrix(y_true, y_pred, 'DistilBERT Confusion Matrix', 'distilbert_confusion_matrix.png')

#Model Comparison Bar Chart
models = ['Logistic Regression', 'CNN', 'DistilBERT']
accuracies = [0.3201, 0.3288, 0.3397]

plt.figure(figsize=(7, 5))
plt.bar(models, accuracies, color=['#4C72B0', '#DD8452', '#55A868'])
plt.ylim(0.0, 0.5)
plt.ylabel('Accuracy')
plt.title('Model Comparison - Fake News (LIAR2) Truthfulness Classification')
for i, acc in enumerate(accuracies):
    plt.text(i, acc + 0.01, f'{acc:.4f}', ha='center')
plt.savefig(os.path.join(plot_dir, 'model_comparison.png'), bbox_inches='tight')
plt.close()

#CNN Training Curves
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('CNN Accuracy Curve')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('CNN Loss Curve')
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(plot_dir, 'cnn_training_curve.png'), bbox_inches='tight')
plt.close()

#DistilBERT Training Curve
log_history = pd.DataFrame(trainer.state.log_history)
train_loss = log_history[log_history['loss'].notna()][['epoch', 'loss']]
eval_loss = log_history[log_history['eval_loss'].notna()][['epoch', 'eval_loss']]

plt.figure(figsize=(7, 5))
plt.plot(train_loss['epoch'], train_loss['loss'], label='Train Loss', marker='o')
plt.plot(eval_loss['epoch'], eval_loss['eval_loss'], label='Val Loss', marker='o')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('DistilBERT Training Curve')
plt.legend()
plt.savefig(os.path.join(plot_dir, 'distilbert_training_curve.png'), bbox_inches='tight')
plt.close()

# ROC Curves (one-vs-rest, 6-class) 
def save_roc(y_true, y_score, title, filename):
    y_true_bin = label_binarize(y_true, classes=list(range(n_classes)))
    plt.figure(figsize=(9, 7))
    for i in range(n_classes):
        fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_score[:, i])
        roc_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr, label=f'{labels[i]} (AUC = {roc_auc:.2f})')
    plt.plot([0, 1], [0, 1], 'k--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(title)
    plt.legend()
    plt.savefig(os.path.join(plot_dir, filename), bbox_inches='tight', dpi=150)
    plt.close()

y_score_lr = lr_model.predict_proba(X_test_tfidf)
save_roc(y_test, y_score_lr, 'Logistic Regression ROC Curve', 'lr_roc_curve.png')

y_score_cnn = cnn_model.predict(X_test_pad)
save_roc(y_test_arr, y_score_cnn, 'CNN ROC Curve', 'cnn_roc_curve.png')

y_score_bert = predictions.predictions
y_score_bert = np.exp(y_score_bert) / np.exp(y_score_bert).sum(axis=1, keepdims=True)
save_roc(y_true, y_score_bert, 'DistilBERT ROC Curve', 'distilbert_roc_curve.png')

print(f"All plots saved to the '{plot_dir}' folder.")


save_confusion_matrix(y_test, y_pred_lr, 'Logistic Regression Confusion Matrix', 'lr_confusion_matrix.png')
save_confusion_matrix(y_test_arr, y_pred_cnn, 'CNN Confusion Matrix', 'cnn_confusion_matrix.png')
save_confusion_matrix(y_true, y_pred_bert, 'DistilBERT Confusion Matrix', 'distilbert_confusion_matrix.png')

72/72 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step
All plots saved to the 'plots' folder.


In [22]:
readme_content = """# Fake News Truthfulness Classifier (LIAR2)

A 6-class text classification project that rates political statements on a truthfulness scale, from Pants-fire to True, using the LIAR2 dataset. Three models were built and compared: Logistic Regression, CNN, and DistilBERT.

## Dataset

- **Source:** LIAR2 (`ucsbnlp/liar`), an expanded version of the original LIAR dataset from PolitiFact
- **Training set:** 18,369 statements
- **Test set:** 2,296 statements
- **Classes:** 0 = Pants-fire, 1 = False, 2 = Barely-true, 3 = Half-true, 4 = Mostly-true, 5 = True
- **Note:** Classes are imbalanced (False is most common at 5,284 examples, True is least common at 2,068), reflecting real-world fact-checking patterns

## Models

| Model | Approach |
|---|---|
| Logistic Regression | TF-IDF vectorization (5,000 features) |
| CNN | Embedding + Conv1D + GlobalMaxPooling, with early stopping |
| DistilBERT | Fine-tuned `distilbert-base-uncased`, 2 epochs |

## Results

| Model | Accuracy |
|---|---|
| Logistic Regression | 32.01% |
| CNN | 32.88% |
| DistilBERT | 33.97% |

![Model Comparison](plots/model_comparison.png)

All three models perform similarly, well above the 16.7% random baseline for 6-class classification but far from high accuracy — this reflects the genuine difficulty of judging truthfulness from short statement text alone, without external fact-checking. DistilBERT's precision/recall on the rarest class ("True") was notably weak, highlighting a limitation beyond raw accuracy.

## Project Structure

- lr_model.pkl / tfidf_vectorizer.pkl — Logistic Regression model and vectorizer
- cnn_model.keras / tokenizer.pkl — CNN model and tokenizer
- bert_results/ — DistilBERT training checkpoints (not tracked in GitHub due to size)
- plots/ — All generated plots (confusion matrices, ROC curves, training curves)
- REPORT.md / REPORT.pdf — Full project report

## Requirements

- datasets
- scikit-learn
- tensorflow
- transformers
- evaluate
- matplotlib, seaborn

## Usage

Load any saved model with its corresponding preprocessing tool (vectorizer/tokenizer) to run predictions on new political statements. See REPORT.md for full details, including per-class performance and discussion of the class imbalance issue.
"""

with open("README.md", "w", encoding="utf-8") as f:
    f.write(readme_content)

print("README.md created")

README.md created


In [23]:
report_content = """# Project Report: Fake News Truthfulness Classification (LIAR2)

## 1. Objective

Build and compare three machine learning models to classify political statements on a 6-level truthfulness scale, using the LIAR2 dataset (an expanded version of the LIAR dataset from PolitiFact fact-checks).

## 2. Dataset

- **Source:** LIAR2 (`ucsbnlp/liar`)
- **Training set:** 18,369 statements
- **Test set:** 2,296 statements
- **Fields used:** `statement` (text), `label` (0-5 truthfulness scale)
- **Statement length:** average 17.7 words, max 79 words
- **Class distribution:** Imbalanced - False (5,284) is most common, True (2,068) is least common. Mild-to-moderate skew (~2.5x between largest and smallest class), likely reflecting real-world fact-checking patterns rather than a data quality issue.
- **Data quality:** No missing values in statement or label fields

## 3. Methodology

### 3.1 Logistic Regression
- TF-IDF vectorization (5,000 features, English stop words removed)
- Trained with `sklearn.linear_model.LogisticRegression` (max_iter=1000)

### 3.2 CNN
- Text tokenized and padded to 40 tokens (vocabulary size: 5,000)
- Architecture: Embedding (128 dim) -> Conv1D (128 filters, kernel size 5) -> GlobalMaxPooling1D -> Dense(64) -> Dropout(0.5) -> Dense(6, softmax)
- EarlyStopping applied from the start (patience=2, restore_best_weights=True), stopped at epoch 4, restored epoch 2 weights

![CNN Training Curve](plots/cnn_training_curve.png)

### 3.3 DistilBERT
- `distilbert-base-uncased`, fine-tuned on the full 18,369-row training set
- 2 epochs, batch size 16
- Training took approximately 3.5 hours on CPU
- Validation loss rose slightly between epochs (1.532 -> 1.540) while training loss fell (1.571 -> 1.378), a mild overfitting signal; best-epoch weights were retained

![DistilBERT Training Curve](plots/distilbert_training_curve.png)

## 4. Results

| Model | Accuracy |
|---|---|
| Logistic Regression | 32.01% |
| CNN | 32.88% |
| DistilBERT | **33.97%** |

![Model Comparison](plots/model_comparison.png)

For context: random guessing on 6 classes would score ~16.7%. Published research using statement-only text (no speaker history or external context) for LIAR-style 6-class classification typically reports accuracy in the 20-27% range, so all three models here perform competitively against that baseline.

### 4.1 Logistic Regression

![Logistic Regression Confusion Matrix](plots/lr_confusion_matrix.png)
![Logistic Regression ROC Curve](plots/lr_roc_curve.png)

### 4.2 CNN

![CNN Confusion Matrix](plots/cnn_confusion_matrix.png)
![CNN ROC Curve](plots/cnn_roc_curve.png)

### 4.3 DistilBERT - Classification Report

| Class | Precision | Recall | F1-score |
|---|---|---|---|
| Pants-fire (0) | 0.54 | 0.10 | 0.17 |
| False (1) | 0.41 | 0.69 | 0.52 |
| Barely-true (2) | 0.27 | 0.12 | 0.17 |
| Half-true (3) | 0.28 | 0.27 | 0.27 |
| Mostly-true (4) | 0.25 | 0.43 | 0.31 |
| True (5) | 0.00 | 0.00 | 0.00 |

![DistilBERT Confusion Matrix](plots/distilbert_confusion_matrix.png)
![DistilBERT ROC Curve](plots/distilbert_roc_curve.png)

Notably, DistilBERT never predicted the "True" class correctly (0.00 precision and recall), despite having the highest overall accuracy. This shows the model leaned heavily on the majority class ("False," 69% recall) rather than learning to distinguish all six levels, and illustrates why accuracy alone can be a misleading metric on imbalanced data.

## 5. Discussion

- This is a genuinely difficult task: distinguishing six levels of truthfulness from short statement text alone requires real-world fact verification the models have no access to. Results in the low-to-mid 30% range are consistent with, and slightly above, published baselines for statement-only LIAR-style classification.
- All three models performed similarly (32-34% accuracy), suggesting the bottleneck is information availability in the text itself, not model capacity - a more complex model (DistilBERT) only marginally outperformed a simple TF-IDF baseline.
- Class imbalance (False overrepresented, True underrepresented) likely contributed to DistilBERT's complete failure on the "True" class, despite its best overall accuracy. This highlights the importance of per-class metrics, not just overall accuracy, when evaluating imbalanced classification tasks.
- The middle categories (Barely-true, Half-true, Mostly-true) were consistently the hardest to separate across all models, which makes intuitive sense given how close these judgments are in meaning.

## 6. Conclusion

DistilBERT achieved the highest overall accuracy (33.97%), but its complete failure on the "True" class makes it an imperfect choice without further adjustment (e.g., class weighting). Logistic Regression and CNN performed comparably with more balanced, if generally weaker, per-class behavior. Given the modest performance gap between all three models, Logistic Regression remains a reasonable lightweight choice, while DistilBERT would need class imbalance handling to be reliably deployed.

## 7. Future Work

- Apply class weighting or oversampling to address the "True" class collapse in DistilBERT
- Incorporate additional available fields (speaker history, justification, subject) rather than statement text alone, which published research shows significantly improves accuracy
- Experiment with a 3-class simplification (true/half-true/false) to reduce task difficulty and improve practical usability
- Deploy the best-performing, rebalanced model as a live web app
"""

with open("REPORT.md", "w", encoding="utf-8") as f:
    f.write(report_content)

print("REPORT.md created")

REPORT.md created


In [24]:
from fpdf import FPDF
import os

pdf = FPDF()
pdf.add_page()
pdf.set_font("Helvetica", size=11)

def add_title(text, size=16):
    pdf.set_font("Helvetica", "B", size)
    pdf.multi_cell(0, 10, text)
    pdf.ln(2)
    pdf.set_font("Helvetica", size=11)

def add_section(text):
    pdf.set_font("Helvetica", "B", 13)
    pdf.multi_cell(0, 8, text)
    pdf.ln(1)
    pdf.set_font("Helvetica", size=11)

def add_text(text):
    pdf.multi_cell(0, 6, text)
    pdf.ln(2)

def add_image(path, w=180):
    if os.path.exists(path):
        pdf.image(path, w=w)
        pdf.ln(4)
    else:
        print(f"Missing: {path}")

add_title("Project Report: Fake News Truthfulness Classification (LIAR2)")

add_section("1. Objective")
add_text("Build and compare three machine learning models to classify political statements on a 6-level truthfulness scale, using the LIAR2 dataset from PolitiFact fact-checks.")

add_section("2. Dataset")
add_text("Source: LIAR2 (ucsbnlp/liar)\nTraining set: 18,369 statements\nTest set: 2,296 statements\nClasses are imbalanced - False (5,284) most common, True (2,068) least common.\nAverage statement length: 17.7 words. No missing values.")

add_section("3. Methodology")
add_text("3.1 Logistic Regression - TF-IDF (5,000 features), sklearn LogisticRegression.\n\n3.2 CNN - Embedding + Conv1D + GlobalMaxPooling, early stopping from the start (stopped epoch 4, restored epoch 2).\n\n3.3 DistilBERT - Fine-tuned distilbert-base-uncased, 2 epochs, full 18,369-row dataset, ~3.5 hours on CPU. Mild overfitting observed (val_loss rose slightly).")

add_image("plots/cnn_training_curve.png")
add_image("plots/distilbert_training_curve.png")

add_section("4. Results")
add_text("Logistic Regression: 32.01%\nCNN: 32.88%\nDistilBERT: 33.97%\n\nFor context, random guessing on 6 classes scores ~16.7%. Published statement-only LIAR benchmarks typically report 20-27% accuracy.")
add_image("plots/model_comparison.png")

add_section("4.1 Logistic Regression")
add_image("plots/lr_confusion_matrix.png")
add_image("plots/lr_roc_curve.png")

add_section("4.2 CNN")
add_image("plots/cnn_confusion_matrix.png")
add_image("plots/cnn_roc_curve.png")

add_section("4.3 DistilBERT")
add_text("Notably, DistilBERT never correctly predicted the 'True' class (0.00 precision/recall), despite the highest overall accuracy - it leaned heavily on the majority 'False' class instead.")
add_image("plots/distilbert_confusion_matrix.png")
add_image("plots/distilbert_roc_curve.png")

add_section("5. Discussion")
add_text("This is a genuinely difficult task: distinguishing six truthfulness levels from short text alone requires real-world fact verification the models cannot access. All three models performed similarly, suggesting the bottleneck is information availability, not model capacity. Class imbalance likely contributed to DistilBERT's failure on the 'True' class, underscoring the importance of per-class metrics over overall accuracy alone.")

add_section("6. Conclusion")
add_text("DistilBERT achieved the highest accuracy but failed completely on the 'True' class, making it an imperfect choice without further adjustment. Logistic Regression and CNN performed comparably with more balanced per-class behavior.")

add_section("7. Future Work")
add_text("- Apply class weighting to address the 'True' class collapse\n- Incorporate speaker history, justification, and subject fields\n- Try a simplified 3-class version of the task\n- Deploy the best, rebalanced model as a web app")

pdf.output("REPORT.pdf")
print("REPORT.pdf created successfully")

REPORT.pdf created successfully


In [25]:
import joblib

joblib.dump(lr_model, 'lr_model.pkl')
joblib.dump(vectorizer, 'tfidf_vectorizer.pkl')

['tfidf_vectorizer.pkl']

In [26]:
trainer.save_model('distilbert_model')
bert_tokenizer.save_pretrained('distilbert_model')

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.82it/s]


('distilbert_model\\tokenizer_config.json', 'distilbert_model\\tokenizer.json')

In [27]:
!pip install python-pptx


   ---------------------------------------- 0/2 [XlsxWriter]
   ---------------------------------------- 0/2 [XlsxWriter]
   ---------------------------------------- 0/2 [XlsxWriter]
   ---------------------------------------- 0/2 [XlsxWriter]
   ---------------------------------------- 0/2 [XlsxWriter]
   ---------------------------------------- 0/2 [XlsxWriter]
   ---------------------------------------- 0/2 [XlsxWriter]
   ---------------------------------------- 0/2 [XlsxWriter]
   -------------------- ------------------- 1/2 [python-pptx]
   -------------------- ------------------- 1/2 [python-pptx]
   -------------------- ------------------- 1/2 [python-pptx]
   -------------------- ------------------- 1/2 [python-pptx]
   -------------------- ------------------- 1/2 [python-pptx]
   -------------------- ------------------- 1/2 [python-pptx]
   -------------------- ------------------- 1/2 [python-pptx]
   -------------------- ------------------- 1/2 [python-pptx]
   ------------

In [28]:
from pptx import Presentation
from pptx.util import Inches, Pt
from pptx.dml.color import RGBColor
from pptx.enum.text import PP_ALIGN

prs = Presentation()
prs.slide_width = Inches(13.33)
prs.slide_height = Inches(7.5)

blank_slide_layout = prs.slide_layouts[6]

def add_title_slide(title, subtitle):
    slide = prs.slides.add_slide(blank_slide_layout)
    title_box = slide.shapes.add_textbox(Inches(0.5), Inches(2.5), Inches(12.3), Inches(1.5))
    tf = title_box.text_frame
    tf.text = title
    tf.paragraphs[0].font.size = Pt(40)
    tf.paragraphs[0].font.bold = True
    tf.paragraphs[0].alignment = PP_ALIGN.CENTER

    sub_box = slide.shapes.add_textbox(Inches(0.5), Inches(4.0), Inches(12.3), Inches(1))
    tf2 = sub_box.text_frame
    tf2.text = subtitle
    tf2.paragraphs[0].font.size = Pt(20)
    tf2.paragraphs[0].alignment = PP_ALIGN.CENTER
    return slide

def add_content_slide(title, bullets):
    slide = prs.slides.add_slide(blank_slide_layout)
    title_box = slide.shapes.add_textbox(Inches(0.5), Inches(0.3), Inches(12.3), Inches(0.8))
    tf = title_box.text_frame
    tf.text = title
    tf.paragraphs[0].font.size = Pt(32)
    tf.paragraphs[0].font.bold = True

    body_box = slide.shapes.add_textbox(Inches(0.7), Inches(1.3), Inches(11.9), Inches(5.8))
    tf2 = body_box.text_frame
    tf2.word_wrap = True
    for i, bullet in enumerate(bullets):
        p = tf2.paragraphs[0] if i == 0 else tf2.add_paragraph()
        p.text = f"•  {bullet}"
        p.font.size = Pt(20)
        p.space_after = Pt(14)
    return slide

def add_image_slide(title, image_path, caption=""):
    slide = prs.slides.add_slide(blank_slide_layout)
    title_box = slide.shapes.add_textbox(Inches(0.5), Inches(0.3), Inches(12.3), Inches(0.8))
    tf = title_box.text_frame
    tf.text = title
    tf.paragraphs[0].font.size = Pt(28)
    tf.paragraphs[0].font.bold = True

    if os.path.exists(image_path):
        slide.shapes.add_picture(image_path, Inches(1.5), Inches(1.2), height=Inches(5.5))
    if caption:
        cap_box = slide.shapes.add_textbox(Inches(0.5), Inches(6.9), Inches(12.3), Inches(0.5))
        tf2 = cap_box.text_frame
        tf2.text = caption
        tf2.paragraphs[0].font.size = Pt(14)
        tf2.paragraphs[0].alignment = PP_ALIGN.CENTER
    return slide

import os

# ---- Build the deck ----
add_title_slide(
    "Fake News Truthfulness Classification",
    "LIAR2 Dataset | Logistic Regression, CNN & DistilBERT"
)

add_content_slide("Objective", [
    "Classify political statements on a 6-level truthfulness scale",
    "Dataset: LIAR2, an expanded PolitiFact fact-check dataset",
    "Compare three models: Logistic Regression, CNN, and DistilBERT"
])

add_content_slide("Dataset Overview", [
    "18,369 training statements, 2,296 test statements",
    "6 classes: Pants-fire, False, Barely-true, Half-true, Mostly-true, True",
    "Average statement length: 17.7 words",
    "Classes imbalanced — False most common (5,284), True least common (2,068)"
])

add_content_slide("Methodology", [
    "Logistic Regression: TF-IDF (5,000 features)",
    "CNN: Embedding + Conv1D + GlobalMaxPooling, early stopping applied",
    "DistilBERT: Fine-tuned distilbert-base-uncased, 2 epochs, full dataset"
])

add_content_slide("Results Summary", [
    "Logistic Regression: 32.01% accuracy",
    "CNN: 32.88% accuracy",
    "DistilBERT: 33.97% accuracy",
    "Random baseline (6 classes): ~16.7%",
    "Published statement-only benchmarks: 20-27% accuracy"
])

add_image_slide("Model Comparison", "plots/model_comparison.png")

add_image_slide("CNN Training Curve", "plots/cnn_training_curve.png")
add_image_slide("DistilBERT Training Curve", "plots/distilbert_training_curve.png")

add_image_slide("Logistic Regression — Confusion Matrix", "plots/lr_confusion_matrix.png")
add_image_slide("CNN — Confusion Matrix", "plots/cnn_confusion_matrix.png")
add_image_slide("DistilBERT — Confusion Matrix", "plots/distilbert_confusion_matrix.png")

add_image_slide("DistilBERT — ROC Curve", "plots/distilbert_roc_curve.png")

add_content_slide("Key Finding: The 'True' Class Problem", [
    "DistilBERT scored 0.00 precision and recall on the 'True' class",
    "Despite having the highest overall accuracy",
    "Model leaned on majority class ('False') instead of learning all 6 levels",
    "Highlights why accuracy alone can mislead on imbalanced data"
])

add_content_slide("Discussion", [
    "All models performed similarly — bottleneck is information, not model complexity",
    "Short statements alone lack enough signal for 6-way truthfulness judgments",
    "Class imbalance contributed to DistilBERT's 'True' class failure",
    "Middle categories (Barely/Half/Mostly-true) hardest to separate"
])

add_content_slide("Conclusion & Future Work", [
    "DistilBERT: best accuracy, but unreliable due to class imbalance",
    "Logistic Regression/CNN: more balanced, lower overall accuracy",
    "Next steps: class weighting, add speaker history & justification fields",
    "Consider simplifying to 3-class version for practical deployment"
])

prs.save("Fake_News_Classification_Presentation.pptx")
print("Presentation created successfully")


Presentation created successfully


In [29]:
import os
print(os.path.exists('Fake_News_Classification_Presentation.pptx'))
print(os.path.getsize('Fake_News_Classification_Presentation.pptx'))

True
403507
